# Lab 2: Gravity of Simple Bodies

| | |
|---|---|
| **Module** | M2 — Newtonian Potential and Gravity Fields |
| **Estimated time** | ~2.5 hours |
| **Prerequisites** | Lectures M2-1, M2-2; Homework 2, Problems 1–3 |
| **Textbook** | Blakely Ch. 3–4; lecture notes M2 |

---

## Learning Outcomes

By the end of this lab you will be able to:

1. **Implement** the gravitational potential and vertical attraction of a uniform sphere from first principles
2. **Apply** free-air and Bouguer corrections to a synthetic gravity profile
3. **Compute** $g(r)$ through a layered Earth model by integrating a density profile
4. **Explain** why surface gravity is not simply related to distance from Earth's center

---

## How to use this notebook

Cells are tagged in their first line:

- **`[PROVIDED]`** — run as-is, do not modify
- **`[IMPLEMENT]`** — replace each `raise NotImplementedError` with correct code
- **`[VALIDATE]`** — run to check your work; prints ✓ PASS or ✗ FAIL with diagnostics
- **`[EXPLORE]`** — starting point; modify freely to answer the question above it

Markdown cells with **Your response:** are written-answer questions — write directly
in the notebook. These are graded.

Need a hint? Run `print(hints['key'])` in a code cell.
Hint keys are listed at the start of each Part.


## Background

The gravitational potential of an extended body is found by summing the potential
of each mass element. For a continuous density distribution $\rho(\mathbf{r}')$,

$$
U(\mathbf{r}) = G \int_V \frac{\rho(\mathbf{r}')}{|\mathbf{r} - \mathbf{r}'|}\, dV'
$$

For a **uniform sphere** of radius $R$, density $\rho$, and total mass
$M = \frac{4}{3}\pi R^3 \rho$, the shell theorem (Blakely §3.2) allows an
exact closed-form evaluation:

$$
U(r) = \begin{cases}
  \dfrac{GM}{r} & r \geq R \\[8pt]
  \dfrac{GM}{2R}\left(3 - \dfrac{r^2}{R^2}\right) & r < R
\end{cases}
$$

The **vertical gravitational attraction** (what a gravimeter measures) is
$g_z = \partial U / \partial z$, taken positive downward (Blakely sign convention).
For a body buried at depth $d$ below a horizontal profile, you will need to
account for the geometry carefully.

**Gravity corrections** remove the effect of factors unrelated to subsurface density:
the free-air correction removes the $1/r^2$ dependence on elevation, and the
Bouguer correction removes the attraction of the rock column between the observation
point and the datum. See Blakely §4.1–4.2 for derivations.

In Part 3 we build a radial gravity profile through Earth using the
**Preliminary Reference Earth Model (PREM)**, which gives $\rho(r)$ as a
piecewise polynomial. By integrating $\rho(r)$ in spherical shells we can
compute the enclosed mass $M(r)$ and hence $g(r) = GM(r)/r^2$.


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# Imports and constants. Run this cell first.

import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate

# Physical constants
G       = 6.674e-11    # gravitational constant, N m^2 kg^-2
R_earth = 6.371e6      # mean Earth radius, m
g_surf  = 9.81         # surface gravity reference, m/s^2

# Plotting defaults
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print('Setup complete.')

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# Hint dictionary. Access with:  print(hints['key'])

hints = {
    # Part 1
    'p1_outside': (
        "Outside a uniform sphere (r >= R), the shell theorem says all the mass "
        "acts as if concentrated at the center. The total mass is M = (4/3)*pi*R^3*rho. "
        "See Blakely eq. 3.6."
    ),
    'p1_inside': (
        "Inside the sphere (r < R), only the mass enclosed within radius r contributes "
        "to the attraction — the outer shell cancels. For the potential (not attraction), "
        "you need the full expression from Blakely eq. 3.7."
    ),
    'p1_gz_geometry': (
        "For a sphere whose center is at depth d below the surface, and you are "
        "measuring at horizontal position x along a profile, the distance from "
        "observation point to sphere center is r = sqrt(x^2 + d^2). "
        "The vertical component of gravity is gz = G*M/r^2 * (d/r), "
        "i.e., project the radial field onto the vertical direction."
    ),
    # Part 2
    'p2_free_air': (
        "The free-air correction at elevation h (in meters) is approximately "
        "+0.3086 * h mGal/m. It is ADDED to observed gravity to reduce it to the "
        "datum (sea level). See Blakely eq. 4.1."
    ),
    'p2_bouguer': (
        "The simple Bouguer correction is delta_g_B = 2*pi*G*rho*h "
        "(in SI units). With rho = 2670 kg/m^3 this gives ~0.1119 mGal/m. "
        "It is SUBTRACTED from the free-air corrected anomaly. See Blakely eq. 4.4."
    ),
    # Part 3
    'p3_shells': (
        "Model Earth as nested spherical shells. The enclosed mass within radius r is "
        "M(r) = 4*pi * integral_0^r rho(r') * r'^2 dr'. "
        "Use scipy.integrate.cumulative_trapezoid (or quad) on 4*pi*rho(r)*r^2 "
        "to get M(r) as an array."
    ),
    'p3_prem_poly': (
        "PREM gives density as a polynomial in x = r/R_earth within each layer. "
        "The provided prem_density(r) function handles the piecewise evaluation. "
        "You do not need to re-implement it — just call it."
    ),
}

# Example: print(hints['p1_outside'])

---
## Part 1: Gravitational Field of a Uniform Sphere *(Guided — ~40 min)*

We build up from the scalar potential to the observable vertical attraction,
then model a survey profile over a buried sphere.

**Available hints:** `p1_outside`, `p1_inside`, `p1_gz_geometry`


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 1: Complete sphere_potential. The formula is given in the docstring;
# your job is to turn it into correct NumPy code.

def sphere_potential(r, rho, R):
    """
    Gravitational potential of a uniform sphere at radial distance r.

    Parameters
    ----------
    r : float or ndarray
        Distance from sphere center, meters. Must be > 0.
    rho : float
        Sphere density, kg/m^3.
    R : float
        Sphere radius, meters.

    Returns
    -------
    U : float or ndarray
        Gravitational potential, m^2/s^2 (J/kg).
        Positive convention: U = +GM/r outside (attractive, Blakely eq. 3.6).

    Notes
    -----
    Outside (r >= R):  U = G*M/r,   M = (4/3)*pi*R^3*rho
    Inside  (r <  R):  U = (G*M)/(2*R) * (3 - r^2/R^2)
    """
    r = np.asarray(r, dtype=float)
    U = np.zeros_like(r)

    # Step 1: total mass of the sphere
    # YOUR CODE HERE
    raise NotImplementedError

    # Step 2: outside the sphere (r >= R)
    outside = r >= R
    # YOUR CODE HERE  (set U[outside] = ...)
    raise NotImplementedError

    # Step 3: inside the sphere (r < R)
    inside = r < R
    # YOUR CODE HERE  (set U[inside] = ...)
    raise NotImplementedError

    return U

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────
# Do not modify. Tests sphere_potential against known analytical values.

def _check(label, got, expected, rtol=1e-5, atol=0.0):
    if np.allclose(got, expected, rtol=rtol, atol=atol):
        print(f'  ✓ PASS  {label}')
    else:
        print(f'  ✗ FAIL  {label}')
        print(f'    expected: {np.asarray(expected)}')
        print(f'    got:      {np.asarray(got)}')

_R   = 1000.0        # 1 km radius sphere
_rho = 2700.0        # kg/m^3
_M   = (4/3)*np.pi*_R**3*_rho

# Test 1: potential at r = 2R (outside)
_check('U at r=2R (outside)', sphere_potential(2*_R, _rho, _R), G*_M/(2*_R))

# Test 2: potential is continuous at r = R
_check('U continuous at r=R',
       sphere_potential(_R*(1-1e-8), _rho, _R),
       sphere_potential(_R*(1+1e-8), _rho, _R),
       rtol=1e-4)

# Test 3: inside potential > surface potential (well shape)
U_center  = sphere_potential(1e-3, _rho, _R)   # near center
U_surface = sphere_potential(_R,   _rho, _R)
if U_center > U_surface:
    print('  ✓ PASS  U(center) > U(surface) [potential well]')
else:
    print('  ✗ FAIL  U(center) should be > U(surface)')

# Test 4: vector input
_r_arr = np.array([2*_R, 3*_R, 4*_R])
_check('vector input', sphere_potential(_r_arr, _rho, _R), G*_M/_r_arr)

In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 1: Vertical gravitational attraction of a buried sphere along a profile.

def sphere_gz_profile(x, depth, rho, R):
    """
    Vertical gravitational attraction of a uniform sphere along a horizontal profile.

    Parameters
    ----------
    x : ndarray, shape (N,)
        Horizontal positions along profile, meters (sphere center at x=0).
    depth : float
        Depth to sphere center below the profile, meters. Must be > R.
    rho : float
        Density contrast of sphere with surroundings, kg/m^3.
    R : float
        Sphere radius, meters.

    Returns
    -------
    gz : ndarray, shape (N,)
        Vertical gravity anomaly, m/s^2 (positive downward).

    Notes
    -----
    The sphere is entirely below the profile (depth > R), so we only need
    the exterior formula.  The 3D distance from profile point (x, 0, 0) to
    sphere center (0, 0, depth) is r = sqrt(x^2 + depth^2).
    The vertical component: gz = G*M/r^2 * (depth/r).
    """
    x = np.asarray(x, dtype=float)

    # Step 1: total mass
    # YOUR CODE HERE
    raise NotImplementedError

    # Step 2: distance r from each profile point to sphere center
    # YOUR CODE HERE
    raise NotImplementedError

    # Step 3: vertical component
    # YOUR CODE HERE
    raise NotImplementedError

    return gz

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

_R2    = 500.0       # sphere radius, m
_rho2  = 500.0       # density contrast, kg/m^3
_depth = 2000.0      # depth to center, m
_M2    = (4/3)*np.pi*_R2**3*_rho2

# Test 1: maximum at x=0 equals G*M/depth^2
_expected_peak = G*_M2/_depth**2
_check('peak gz at x=0', sphere_gz_profile([0.0], _rho2, _R2, _depth)[0],  # intentional wrong arg order to catch order bugs
       _expected_peak, rtol=1e-5)

# Correct arg order version (this one should pass if your function is correct)
_check('peak gz at x=0 (correct order)',
       sphere_gz_profile(np.array([0.0]), _depth, _rho2, _R2)[0],
       _expected_peak, rtol=1e-5)

# Test 2: symmetric profile (gz[x] == gz[-x])
_x = np.linspace(-5000, 5000, 201)
_gz = sphere_gz_profile(_x, _depth, _rho2, _R2)
if np.allclose(_gz, _gz[::-1], rtol=1e-10):
    print('  ✓ PASS  Profile is symmetric about x=0')
else:
    print('  ✗ FAIL  Profile should be symmetric about x=0')

# Test 3: gz → 0 far from the sphere
_check('gz → 0 at large x',
       sphere_gz_profile(np.array([1e8]), _depth, _rho2, _R2)[0],
       0.0, atol=1e-12)

### Question 1.1 — Shape of the anomaly

The gravity anomaly of a buried sphere has a characteristic bell shape.
The **half-width** of the anomaly $x_{1/2}$ (the horizontal distance at which
$g_z$ drops to half its peak value) is often used in exploration geophysics
to estimate depth.

Using your `sphere_gz_profile` function, find $x_{1/2}$ numerically for the
sphere defined above ($d = 2000$ m, $R = 500$ m). Then derive analytically
what $x_{1/2}$ should be in terms of $d$ (hint: set $g_z(x_{1/2}) = g_z(0)/2$
and solve). How close is your numerical answer to the analytical formula?

**Your response:**

> *(Write 3–5 sentences. Include both your numerical estimate and the analytical result.)*


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Find x_{1/2} numerically and plot the profile.
# Label the half-width on your plot.

x = np.linspace(-8000, 8000, 400)   # m
depth = 2000.0   # m
rho_contrast = 500.0   # kg/m^3
R_sphere = 500.0   # m

# YOUR CODE HERE

### Question 1.2 — Depth vs. anomaly amplitude trade-off

A smaller, denser sphere buried at the same depth produces an anomaly with
the same peak amplitude as a larger, less dense sphere.

Construct two spheres that produce the same peak $g_z$ at the surface but have
different depths. Plot both profiles on the same axes. What observable feature
of the profiles would allow you to distinguish between them? What would remain
ambiguous?

**Your response:**

> *(Write 4–6 sentences. This is the basis of the "gravity ambiguity" problem.)*


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Construct your two equivalent-amplitude spheres and plot their profiles.
# Requirements:
#   - Both profiles on same axes, different colors, labeled
#   - x-axis: horizontal distance, km
#   - y-axis: gz in mGal  (1 mGal = 1e-5 m/s^2)

# YOUR CODE HERE

---
## Part 2: Gravity Corrections *(Supported — ~50 min)*

A real gravity survey measures the total gravity at each station, which includes
contributions from elevation differences, terrain, and tidal effects — not just
the subsurface density we care about. We apply **corrections** to isolate the
anomaly of interest.

In this Part you will apply free-air and Bouguer corrections to a synthetic
gravity profile collected over uneven terrain.

**Available hints:** `p2_free_air`, `p2_bouguer`


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# Synthetic survey data. Run as-is.
#
# Scenario: a 50-km profile across a ridge. The profile contains:
#   - A Gaussian topographic ridge (h_max = 800 m, sigma = 8 km)
#   - A buried sphere (positive density contrast) beneath the ridge
#   - Random noise (sigma = 0.05 mGal)

rng = np.random.default_rng(2025)

# Profile coordinates
x_km  = np.linspace(-25, 25, 251)    # km
x_m   = x_km * 1e3                  # m

# Terrain elevation (meters above sea level)
h_terrain = 800.0 * np.exp(-(x_m**2) / (2*(8e3)**2))

# True buried sphere (students don't know these values — they are for generating data)
_true_depth   = 5000.0   # m below surface
_true_R       = 1500.0   # m
_true_rho     = 300.0    # kg/m^3 density contrast

# Sphere anomaly evaluated at ground surface (z=0 datum), ignoring terrain
_sphere_signal = sphere_gz_profile(x_m, _true_depth, _true_rho, _true_R)  # m/s^2
_sphere_signal_mGal = _sphere_signal * 1e5   # convert to mGal

# Free-air effect of terrain (approximate)
_FA_rate = 0.3086e-3   # mGal/m (free-air gradient)
_fa_effect = _FA_rate * h_terrain

# Bouguer effect of terrain slab (approximate)
_rho_crust = 2670.0    # kg/m^3
_bg_rate   = 2 * np.pi * G * _rho_crust * 1e5   # mGal/m
_bg_effect = _bg_rate * h_terrain

# "Observed" gravity = normal gravity + terrain effects + sphere signal + noise
g_normal = 980000.0   # mGal (approximate surface gravity in mGal)
noise    = rng.normal(0, 0.05, size=x_km.shape)

g_observed = g_normal + _fa_effect - _bg_effect + _sphere_signal_mGal + noise

print(f'Profile: {len(x_km)} stations over {x_km[-1]-x_km[0]:.0f} km')
print(f'Terrain relief: {h_terrain.max():.0f} m')
print(f'Observed gravity range: {g_observed.min():.1f} – {g_observed.max():.1f} mGal')

In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 2: implement the free-air correction and the simple Bouguer correction.
# No step-by-step skeleton is provided. Read the docstrings and use hints if needed.

def free_air_correction(h):
    """
    Free-air correction to reduce observed gravity to sea-level datum.

    Parameters
    ----------
    h : float or ndarray
        Station elevation above datum, meters.

    Returns
    -------
    delta_g_FA : float or ndarray
        Correction in mGal. ADD this to observed gravity.

    Notes
    -----
    The linear approximation is: delta_g_FA ≈ +0.3086 * h  [mGal],
    where h is elevation in meters (Blakely eq. 4.1, simplified).
    """
    raise NotImplementedError


def bouguer_correction(h, rho_slab=2670.0):
    """
    Simple Bouguer correction (infinite slab approximation).

    Parameters
    ----------
    h : float or ndarray
        Station elevation above datum, meters.
    rho_slab : float, optional
        Assumed density of the rock slab, kg/m^3.  Default: 2670.

    Returns
    -------
    delta_g_B : float or ndarray
        Correction in mGal. SUBTRACT this from the free-air anomaly.

    Notes
    -----
    The Bouguer slab correction is: delta_g_B = 2*pi*G*rho_slab*h
    In SI units this is m/s^2; convert to mGal (1 mGal = 1e-5 m/s^2).
    (Blakely eq. 4.4)
    """
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

# Free-air: at h=1000 m correction should be ~308.6 mGal
_check('FA correction at 1000 m', free_air_correction(1000.0), 308.6, rtol=1e-3)

# Bouguer: at h=1000 m, rho=2670, correction ≈ 111.9 mGal
_check('Bouguer correction at 1000 m',
       bouguer_correction(1000.0, 2670.0), 111.9, rtol=1e-2)

# Both corrections should be zero at h=0
_check('FA correction at h=0', free_air_correction(0.0), 0.0, atol=1e-10)
_check('Bouguer correction at h=0', bouguer_correction(0.0), 0.0, atol=1e-10)

# Bouguer correction should increase with density
_bg_low  = bouguer_correction(100.0, 2000.0)
_bg_high = bouguer_correction(100.0, 3000.0)
if _bg_high > _bg_low:
    print('  ✓ PASS  Bouguer correction increases with density')
else:
    print('  ✗ FAIL  Bouguer correction should increase with density')

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Apply your corrections to g_observed and compute:
#   1. The free-air anomaly:  g_FA = g_observed + FA_correction - g_normal
#   2. The Bouguer anomaly:   g_BA = g_FA - Bouguer_correction
#
# Then make a 3-panel figure:
#   Panel 1: terrain elevation h_terrain vs x_km
#   Panel 2: free-air anomaly vs x_km
#   Panel 3: Bouguer anomaly vs x_km
#
# All gravity axes should be in mGal; label units on all axes.
# Add a dashed vertical line at x=0 on each panel (location of sphere center).

# YOUR CODE HERE

### Question 2.1 — What the Bouguer anomaly tells you

After applying corrections, is the buried sphere visible in the Bouguer anomaly?
Describe what you see and whether it matches your expectation from Part 1.
Why is the Bouguer anomaly a better starting point for interpreting subsurface
structure than the raw observed gravity?

**Your response:**

> *(Write 4–5 sentences.)*


### Question 2.2 — Sensitivity to assumed density

The Bouguer correction depends on an assumed slab density. Recompute the
Bouguer anomaly using $\rho = 2000$ kg/m³ and $\rho = 3300$ kg/m³.
How much does the choice of density affect the anomaly amplitude and shape?
What does this imply for real survey interpretation?

**Your response:**

> *(Write 3–5 sentences. You may include a figure.)*


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Test two different assumed densities for the Bouguer correction.

# YOUR CODE HERE

---
## Part 3: Gravity Through a Layered Earth *(Open — ~40 min)*

How does $g(r)$ vary with depth inside the Earth? A naive guess might be
"it decreases monotonically from surface to center" — but the PREM density
model tells a different story.

Your goal: compute and plot $g(r)$ from $r = 0$ to $r = R_{\oplus}$ using
the PREM density profile provided below.

**Available hints:** `p3_shells`, `p3_prem_poly`


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# PREM density model (Dziewonski & Anderson 1981), simplified piecewise polynomial.
# Density is in kg/m^3 as a function of r in meters.

def prem_density(r):
    """
    PREM density profile (simplified), evaluated at radius r (meters).

    Returns density in kg/m^3. Piecewise polynomial in x = r/R_earth.
    Based on Dziewonski & Anderson (1981), Table 1.
    """
    r = np.asarray(r, dtype=float)
    x = r / R_earth
    rho = np.zeros_like(x)

    # Inner core (r < 1221.5 km)
    m = x < 0.1917
    rho[m] = 1000 * (13.0885 - 8.8381 * x[m]**2)

    # Outer core (1221.5 – 3480 km)
    m = (x >= 0.1917) & (x < 0.5466)
    rho[m] = 1000 * (12.5815 - 1.2638*x[m] - 3.6426*x[m]**2 - 5.5281*x[m]**3)

    # Lower mantle (3480 – 5701 km)
    m = (x >= 0.5466) & (x < 0.8952)
    rho[m] = 1000 * (7.9565 - 6.4761*x[m] + 5.5283*x[m]**2 - 3.0807*x[m]**3)

    # Transition zone + upper mantle (5701 – 6346 km)
    m = (x >= 0.8952) & (x < 0.9965)
    rho[m] = 1000 * (11.2494 - 8.0298*x[m])

    # Crust (6346 – 6371 km) — simplified single layer
    m = x >= 0.9965
    rho[m] = 2900.0

    return rho

# Quick sanity check
print(f'PREM density at center:  {prem_density(0):.0f} kg/m^3  (expect ~13000)')
print(f'PREM density at surface: {prem_density(R_earth):.0f} kg/m^3  (expect ~2900)')
print(f'PREM density at CMB:     {prem_density(3.48e6):.0f} kg/m^3  (expect ~5500–9900)')

### Task 3.1 — Compute and plot $g(r)$

Using `prem_density`, compute and plot $g(r)$ from $r = 0$ to $r = R_{\oplus}$.
Also plot the density profile on a second y-axis or a separate panel.

For comparison, also plot $g(r)$ for a **uniform Earth** with the same total
mass as PREM (you will need to compute the mean density first).

Requirements for your plot:
- x-axis: radius in km (0 to 6371)
- y-axis: $g$ in m/s²
- Label major Earth layers (inner core, outer core, mantle, crust) with vertical lines or a shaded band
- Both the PREM and uniform-Earth curves on the same axes
- Your computed $g(R_{\oplus})$ should be within 1% of the standard value of 9.81 m/s²


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Compute g(r) for PREM and for a uniform Earth.
# Hint: M(r) = 4*pi * integral_0^r rho(r') * r'^2 dr'
# then g(r) = G*M(r)/r^2 (for r > 0; g(0) = 0 by symmetry).

# Define a radial grid (avoid r=0 to prevent division by zero)
r_grid = np.linspace(1e3, R_earth, 1000)   # meters

# YOUR CODE HERE

### Question 3.1 — The gravity maximum

In the PREM model, $g(r)$ is not monotonically decreasing from surface to center.
Where approximately is the maximum? Which major layer boundary is it near?
Explain physically why $g$ can *increase* as you go deeper into Earth.

**Your response:**

> *(Write 4–6 sentences. Reference the density profile in your explanation.)*


### Question 3.2 — Uniform Earth comparison

How does the uniform-Earth profile differ from PREM? In which regions is the
uniform approximation worst? What does this tell you about the importance of
density stratification for gravity modeling?

**Your response:**

> *(Write 3–5 sentences.)*


---
## Synthesis

### S1 — Connecting Parts 1 and 3

In Part 1 you computed the anomaly of a *density contrast* (sphere minus host).
In Part 3 you computed $g(r)$ due to total density (absolute, not contrast).
Explain the distinction. If you buried the sphere from Part 1 at the
core–mantle boundary, how would you set up the density contrast? What reference
would you subtract from?

**Your response:**

> *(Write 4–6 sentences.)*

### S2 — Limitations of the infinite-slab Bouguer correction

The simple Bouguer correction assumes an infinite horizontal slab. Where does
this approximation break down in practice? Name two situations where you would
need to apply a **terrain correction** instead, and describe qualitatively what
error the slab approximation introduces in those cases.

**Your response:**

> *(Write 4–6 sentences.)*

### S3 — Uniqueness and the sphere

You showed in Question 1.2 that different spheres can produce identical gravity
profiles. This is an instance of the **non-uniqueness of potential field inversion**.
Given only the Bouguer anomaly from Part 2, what additional information (data or
constraints) would allow you to uniquely determine the depth of the buried sphere?

**Your response:**

> *(Write 4–6 sentences.)*


---
## Extensions *(optional)*

### E1 — Vertical cylinder

Derive the vertical gravity attraction of an infinite vertical cylinder of radius
$R$ and density contrast $\Delta\rho$, then implement it and compare its
profile shape to a sphere with the same peak anomaly. Which decays faster with
distance? Why?

### E2 — PREM total mass

Integrate PREM to compute the total mass of the Earth and compare to the
accepted value $M_{\oplus} = 5.972 \times 10^{24}$ kg. How accurate is this
simplified PREM parameterization? Which layer contributes most to the total mass?

### E3 — Nafe–Drake and estimating Bouguer density

The optimal Bouguer density can be estimated from the correlation between
topography and free-air anomaly (Nettleton's method). Implement a simple
version of this for the synthetic profile in Part 2: compute the Bouguer
anomaly for a range of assumed densities and find the one that minimizes the
correlation with topography. How close is the "optimal" density to $2670$ kg/m³?


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Extension workspace.

# YOUR CODE HERE